# PHẦN 1: CÀI ĐẶT & IMPORT THƯ VIỆN

In [ ]:
!pip install -q transformers tqdm matplotlib seaborn scikit-learn pandas

import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import Counter
from sklearn.manifold import TSNE
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel

# PHẦN 2: KHỞI TẠO DATASET VÀ MÔ HÌNH

In [ ]:
IMAGE_DIR = "/kaggle/input/datasets/vanton2005/oxford-pet-data/images/images" 
ANNOTATION_FILE = "/kaggle/input/datasets/vanton2005/oxford-pet-data/annotations/annotations/list.txt"

class OxfordPetDataset(Dataset):
    def __init__(self, image_dir, annotation_file, processor):
        self.image_dir = image_dir
        self.processor = processor
        self.data = []
        self.classes = []

        with open(annotation_file, 'r') as f:
            for line in f:
                if line.startswith('#'): continue
                parts = line.strip().split()
                if parts:
                    image_name = parts[0]
                    breed = "_".join(image_name.split('_')[:-1]).replace('_', ' ')
                    self.data.append({
                        'image_path': os.path.join(image_dir, f"{image_name}.jpg"),
                        'breed': breed
                    })
                    if breed not in self.classes:
                        self.classes.append(breed)
        
        self.classes = sorted(self.classes)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if not os.path.exists(item['image_path']):
            raise FileNotFoundError(f"Không tìm thấy ảnh: {item['image_path']}")
            
        image = Image.open(item['image_path']).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'label': item['breed']
        }

# Khởi tạo mô hình & Processor
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang chạy trên thiết bị: {device}")

model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

dataset = OxfordPetDataset(IMAGE_DIR, ANNOTATION_FILE, processor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2)

# PHẦN 3: EDA - TRỰC QUAN HÓA DATASET

In [ ]:
print("\n--- 3.1. THỐNG KÊ DATASET ---")
all_labels_in_data = [item['breed'] for item in dataset.data]
class_counts = Counter(all_labels_in_data)
df_counts = pd.DataFrame.from_dict(class_counts, orient='index', columns=['count']).reset_index()
df_counts = df_counts.rename(columns={'index': 'breed'}).sort_values(by='count', ascending=False)

print(f"Tổng số ảnh: {len(dataset)} | Tổng số classes: {len(dataset.classes)}")

# Biểu đồ phân bố nhãn
plt.figure(figsize=(16, 5))
sns.barplot(data=df_counts, x='breed', y='count', palette='viridis')
plt.xticks(rotation=90)
plt.title("Phân bố số lượng ảnh theo từng giống chó/mèo (Oxford-IIIT Pet)")
plt.tight_layout()
plt.show()

In [ ]:
print("\n--- 3.2. HIỂN THỊ ẢNH MẪU ---")
plt.figure(figsize=(8, 8))
for i in range(9):
    idx = random.randint(0, len(dataset) - 1)
    item = dataset.data[idx]
    img = Image.open(item['image_path']).convert("RGB")
    ax = plt.subplot(3, 3, i + 1)
    ax.imshow(img)
    ax.set_title(item['breed'], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

# PHẦN 4: TRÍCH XUẤT ĐẶC TRƯNG & ĐÁNH GIÁ

In [ ]:
print("\n--- 4. TRÍCH XUẤT ĐẶC TRƯNG & ĐÁNH GIÁ (ZERO-SHOT) ---")
# Prompt Engineering
prompts = [f"Picture of a pet with breed: {c}, which is a type of cat/dog" for c in dataset.classes]
text_inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)

model.eval()
with torch.no_grad():
    # Trích xuất Text Features
    text_outputs = model.text_model(**text_inputs)
    text_features = model.text_projection(text_outputs.pooler_output)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

correct_predictions, total_predictions = 0, 0
all_image_features, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Extracting & Evaluating"):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['label']
        
        # Trích xuất Image Features
        image_outputs = model.vision_model(pixel_values=pixel_values)
        image_features = model.visual_projection(image_outputs.pooler_output)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        all_image_features.append(image_features.cpu())
        all_labels.extend(labels)
        
        # Zero-shot classification
        logit_scale = model.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.t()
        predictions = logits_per_image.argmax(dim=1)
        
        for i, pred_idx in enumerate(predictions):
            pred_breed = dataset.classes[pred_idx.item()]
            if pred_breed == labels[i]:
                correct_predictions += 1
            total_predictions += 1

accuracy = (correct_predictions / total_predictions) * 100
print(f"\n=> Độ chính xác Zero-shot: {accuracy:.2f}%")

# Lưu Features
torch.save({
    'image_features': torch.cat(all_image_features),
    'labels': all_labels,
    'classes': dataset.classes
}, '/kaggle/working/extracted_features.pt')
print("Đã lưu Image Features vào file extracted_features.pt!")

# PHẦN 5: TRỰC QUAN HÓA KHÔNG GIAN ĐẶC TRƯNG (t-SNE)

In [ ]:
print("\n--- 5. TRỰC QUAN HÓA FEATURES VỚI t-SNE ---")
features_np = torch.cat(all_image_features).cpu().numpy()
labels_np = np.array(all_labels)

# Chọn ngẫu nhiên 10 class để biểu đồ không bị rối
np.random.seed(42)
selected_classes = np.random.choice(dataset.classes, size=10, replace=False)
mask = np.isin(labels_np, selected_classes)

filtered_features = features_np[mask]
filtered_labels = labels_np[mask]

# Chạy t-SNE nén từ 512D xuống 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
reduced_features = tsne.fit_transform(filtered_features)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=reduced_features[:, 0], 
    y=reduced_features[:, 1], 
    hue=filtered_labels, 
    palette="tab10", 
    s=60, alpha=0.8
)
plt.title("Không gian đặc trưng 2D của CLIP (10 Classes) - t-SNE")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()